<a href="https://colab.research.google.com/github/nicoavilan/Clases/blob/main/Taller_Iniciacion_Qiskit_Fall_Fest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iniciación a Qiskit: tu primer programa en un computador cuántico real

**Qiskit Fall Fest 2026** · Taller práctico

Universidad Distrital Francisco José de Caldas

Professor: [Nicolás Avilán Vargas](http://www.linkedin.com/in/nicoavilanv)

nicolasg.avilan@urosario.edu.co

**Universidad del Rosario - Escuela de Ciencias e Ingeniería**

**Matemáticas Aplicadas y Ciencias de la Computación**

En este taller daremos los primeros pasos en computación cuántica usando **Qiskit**. Partiremos de los conceptos esenciales (qubits, superposición, entrelazamiento y medición) y construiremos un **estado de Bell**, el «hola mundo» cuántico. Lo ejecutaremos en un simulador y luego en un **procesador cuántico real de IBM**, comparando los resultados para entender el efecto del ruido. Exploraremos la **transpilación** y las primitivas **Sampler** y **Estimator**, y calcularemos el valor esperado de un **Hamiltoniano** sencillo.

No se requiere experiencia previa en física cuántica: solo curiosidad y Python básico.


## 0. Preparación del entorno

Si estás en Google Colab, ejecuta la siguiente celda una sola vez (tarda alrededor de un minuto). Si trabajas en tu computador y ya tienes Qiskit instalado, puedes omitirla.

In [ ]:
# Instalación de Qiskit, el proveedor de IBM Runtime y el simulador Aer.
# Fijamos qiskit-ibm-runtime < 0.50 para mantener compatibilidad con este cuaderno.
%pip install 'qiskit[visualization]' "qiskit-ibm-runtime<0.50" qiskit-aer --quiet

Este cuaderno funciona en la versión de 2.6.1 de Qiskit

In [ ]:
# Verificamos las versiones instaladas: útil si algo falla y hay que pedir ayuda.
import qiskit
import qiskit_aer  # Simulador para ejecutar circuitos cuánticos
import qiskit_ibm_runtime # Conecta con el computador cuántico real en la nube

print("qiskit:            ", qiskit.__version__)
print("qiskit-aer:        ", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)

In [ ]:
# Importaciones principales que usaremos a lo largo del taller
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator

## 1. Conceptos esenciales: qubits, superposición y medición

### 1.1 El qubit

Un bit clásico vale 0 o 1. Un **qubit** se describe con un vector de estado

$$
|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle, \qquad \alpha,\beta\in\mathbb{C},\quad |\alpha|^2 + |\beta|^2 = 1 .
$$

Los números $\alpha$ y $\beta$ se llaman **amplitudes**. No los podemos leer directamente: cuando **medimos**, obtenemos 0 con probabilidad $|\alpha|^2$ y 1 con probabilidad $|\beta|^2$ (regla de Born), y el estado *colapsa* al resultado obtenido.

En Qiskit, todo qubit empieza en $|0\rangle$. Veamos su vector de estado y su representación en la **esfera de Bloch**.

In [ ]:
# Un circuito con un solo qubit y sin compuertas: el qubit está en |0⟩
qc = QuantumCircuit(1)

estado = Statevector(qc)
display(estado.draw("latex"))      # vector de estado en notación de Dirac
print("Probabilidades de medir 0 y 1:", estado.probabilities())
plot_bloch_multivector(estado)     # el estado |0⟩ apunta al polo norte

### 1.2 Superposición: la compuerta de Hadamard

La compuerta **H** (Hadamard) transforma $|0\rangle$ en

$$
|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}},
$$

una **superposición** con igual peso en 0 y en 1.

Ojo: no significa que el qubit «sea 0 y 1 a la vez», sino que está en un estado nuevo, $|+\rangle$, que al medirlo da 0 o 1 con 50 % de probabilidad cada uno; a diferencia de una moneda lanzada al aire, las amplitudes del  qubit pueden sumarse o cancelarse como ondas, y eso es lo que llamamos **interferencia**.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)   # Hadamard sobre el qubit 0

qc.draw("mpl") # Dibujo el circuito

In [ ]:
estado = Statevector(qc)
display(estado.draw("latex"))
print("Probabilidades de medir 0 y 1:", estado.probabilities())
plot_bloch_multivector(estado)     # ahora el vector apunta sobre el ecuador (eje x)

### 1.3 Medición

¿Cómo observo una distribución de probabilidad?

Si medimos $|+\rangle$ muchas veces (cada repetición se llama **shot**), esperamos ver aproximadamente la mitad de 0 y la mitad de 1. Podemos simular el muestreo directamente desde el vector de estado:

In [ ]:
# Muestreamos 1000 mediciones simuladas del estado |+⟩
conteos = {str(k): int(v) for k, v in estado.sample_counts(shots=1000).items()}
print(conteos)
plot_histogram(conteos, title="Medición de |+⟩ (1000 shots)")

### 1.4 Interferencia: ¿qué pasa si aplicamos H dos veces?

$H$ es su propia inversa: $H\,H = I$. Las amplitudes de los dos «caminos» que llevan a $|1\rangle$ se **cancelan** (interferencia destructiva) y volvemos a $|0\rangle$ con certeza. Este fenómeno, imposible de imitar con monedas al aire, es el motor de los algoritmos cuánticos.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
display(qc.draw("mpl")) # Dibujo el circuito

estado = Statevector(qc)
display(estado.draw("latex"))
print("Probabilidades tras H·H:", estado.probabilities())

> **Pregunta rápida:** ¿qué estado obtienes si aplicas `qc.x(0)` antes de la Hadamard? ¿Cambian las probabilidades de medición? ¿Cambia el estado? (Pista: usa `Statevector(qc).draw("latex")`).

## 2. Entrelazamiento: el estado de Bell

Con dos qubits, el estado general vive en un espacio de dimensión 4, con base $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$.

> **Convención de Qiskit:** en $|q_1 q_0\rangle$ el qubit 0 se escribe **a la derecha** (orden *little-endian*). Así, el resultado `"10"` significa $q_1 = 1$ y $q_0 = 0$.

El circuito de Bell tiene solo dos compuertas:

1. **H** sobre $q_0$: crea la superposición $\tfrac{1}{\sqrt2}(|00\rangle + |01\rangle)$.
2. **CNOT** (`cx`) con control $q_0$ y objetivo $q_1$: invierte $q_1$ solo si $q_0=1$.

El resultado es

$$
|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}} .
$$

Este estado está **entrelazado**: no puede escribirse como producto de un estado de $q_0$ por uno de $q_1$. Cada qubit, por separado, da 0 o 1 al azar, pero los dos resultados **siempre coinciden**.

In [ ]:
# --- Construimos el circuito de Bell |Φ+⟩ ---
bell = QuantumCircuit(2)
bell.h(0)        # superposición en q0
bell.cx(0, 1)    # entrelazamiento: q0 controla a q1
display(bell.draw("mpl")) # Dibujo el circuito

In [ ]:
# Antes de medir, inspeccionamos el vector de estado
estado_bell = Statevector(bell)
display(estado_bell.draw("latex"))
print("Probabilidades por resultado:", {str(k): round(float(v), 3) for k, v in estado_bell.probabilities_dict().items()})

In [ ]:
# Añadimos las mediciones. measure_all() crea un registro clásico llamado "meas"
bell.measure_all()
bell.draw("mpl")

> **¿Por qué la esfera de Bloch no ayuda aquí?** Si dibujas `plot_bloch_multivector(Statevector(...))` para el estado de Bell, verás ambos vectores en el **centro** de su esfera: cada qubit, mirado por separado, está en un estado completamente mezclado. Toda la información está en las **correlaciones** entre ellos. ¡Inténtalo!

In [ ]:
#

## 3. Ejecución en un simulador

Qiskit ofrece **primitivas**, que son las interfaces estándar para ejecutar circuitos:

| Primitiva | Entrada | Salida |
|-----------|---------|--------|
| **Sampler** | circuito con mediciones | conteos / muestras de bits |
| **Estimator** | circuito + observable | valor esperado $\langle O \rangle$ |

Escribimos una función que sirve igual para el simulador y para el hardware real. El patrón tiene tres pasos que repetiremos todo el taller: **transpilar → ejecutar → leer resultados**.

In [ ]:
def ejecutar_y_contar(circuito, backend, shots=1000):
    """
    Ejecuta un circuito en el backend indicado y devuelve los conteos de medición.

    Parámetros
    ----------
    circuito : QuantumCircuit
        Circuito con mediciones (registro clásico "meas").
    backend :
        Simulador (AerSimulator) o QPU real de IBM.
    shots : int
        Número de repeticiones del circuito.

    Retorna
    -------
    dict
        Diccionario {cadena_de_bits: número_de_veces}.
    """
    # 1) Transpilar: adaptar el circuito a las compuertas y conectividad del backend
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1) # optimización ligera
    circuito_isa = pm.run(circuito)

    # 2) Ejecutar con la primitiva Sampler
    sampler = Sampler(mode=backend)
    trabajo = sampler.run([circuito_isa], shots=shots)
    print("ID del trabajo:", trabajo.job_id())

    # 3) Leer los resultados (esto espera a que el trabajo termine)
    resultado = trabajo.result()
    return resultado[0].data.meas.get_counts()

In [ ]:
# Simulador ideal (sin ruido) que corre en nuestro propio computador
simulador = AerSimulator()

conteos_sim = ejecutar_y_contar(bell, simulador, shots=1000)
print(conteos_sim)
plot_histogram(conteos_sim, title="Estado de Bell en el simulador ideal")

Solo aparecen `00` y `11`, en proporciones cercanas a 50/50. Las pequeñas diferencias vienen del muestreo finito (*ruido estadístico*), no de errores en el «hardware».

## 4. Conexión a IBM Quantum

Para usar un computador cuántico real necesitas una cuenta gratuita en **[IBM Quantum Platform](https://quantum.cloud.ibm.com/)**. El plan abierto (*Open Plan*) incluye algunos minutos de uso de QPU al mes, más que suficientes para este taller.

Necesitas dos datos de tu panel de IBM Quantum:

1. **API key (token):** se crea en el panel principal. Trátala como una contraseña.
2. **Instancia:** el nombre o el CRN de tu instancia (aparece en la sección *Instances*).

> ⚠️ **Seguridad:** nunca escribas tu token directamente en el cuaderno ni lo subas a GitHub. La siguiente celda lo pide con `getpass`, que no lo muestra en pantalla, y lo guarda en tu configuración local. Solo tienes que hacerlo **una vez** por computador (en Colab, una vez por sesión).

In [ ]:
from getpass import getpass

try:
    # Si ya guardaste tus credenciales antes, esto basta
    service = QiskitRuntimeService()
    print("Credenciales encontradas ✔")
except Exception:
    # Primera vez: pedimos el token y la instancia y los guardamos
    token = getpass("Pega aquí tu API key de IBM Quantum: ")
    instancia = input("Nombre o CRN de tu instancia: ").strip()
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        instance=instancia,
        overwrite=True,
        set_as_default=True,
    )
    service = QiskitRuntimeService()
    print("Credenciales guardadas ✔")

In [ ]:
# Listamos las QPU a las que tenemos acceso, con su número de qubits y su cola
for b in service.backends(simulator=False, operational=True):
    print(f"{b.name:20s} qubits: {b.num_qubits:4d}   trabajos en cola: {b.status().pending_jobs}")

In [ ]:
# Elegimos la QPU menos ocupada (o descomentamos una específica, por ejemplo "ibm_fez")
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
# backend = service.backend("ibm_fez")
# backend = service.backend("ibm_marrakesh")

print("Backend seleccionado:", backend.name, "con", backend.num_qubits, "qubits")

> **¿Sin cuenta o sin conexión?** Puedes seguir el taller con un **backend falso** (*fake backend*): una copia local que imita la conectividad, las compuertas y el ruido de una QPU real. Descomenta la siguiente celda y continúa normalmente.

In [ ]:
# --- Alternativa sin cuenta: backend falso que imita a ibm_fez ---
# from qiskit_ibm_runtime.fake_provider import FakeFez
# backend = FakeFez()
# print("Backend falso:", backend.name, "con", backend.num_qubits, "qubits")

## 5. Transpilación: del circuito abstracto al hardware

El circuito que dibujamos es **abstracto**: supone que cualquier qubit puede interactuar con cualquier otro y que existen compuertas como H o CNOT. Una QPU real tiene:

- un **conjunto de compuertas nativas** (*basis gates*), por ejemplo `rz`, `sx`, `x` y `cz`;
- una **conectividad** limitada: solo ciertos pares de qubits físicos pueden interactuar directamente (el *coupling map*);
- qubits con **calidades distintas** (tiempos de coherencia, tasas de error).

La **transpilación** reescribe el circuito para que cumpla esas restricciones. El resultado se llama circuito **ISA** (*Instruction Set Architecture*). Las primitivas de IBM Runtime **solo aceptan circuitos ISA**.

El parámetro `optimization_level` (0 a 3) controla cuánto esfuerzo se dedica a reducir el número de compuertas y a escoger buenos qubits físicos.

In [ ]:
print("Compuertas nativas del backend:", backend.operation_names)

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
bell_isa = pm.run(bell)

# idle_wires=False oculta los ~150 qubits físicos que no usamos
bell_isa.draw("mpl", idle_wires=False)

In [ ]:
# Comparamos el circuito original con el transpilado
print("Original    -> profundidad:", bell.depth(), "| compuertas:", dict(bell.count_ops()))
print("Transpilado -> profundidad:", bell_isa.depth(), "| compuertas:", dict(bell_isa.count_ops()))

# ¿A qué qubits físicos se asignaron nuestros qubits virtuales q0 y q1?
print("Qubits físicos usados:", bell_isa.layout.final_index_layout())

> **Observa:** la Hadamard y la CNOT desaparecen y en su lugar aparecen rotaciones `rz`, raíces de NOT (`sx`) y una compuerta de dos qubits nativa (`cz`). El circuito es **matemáticamente equivalente**, pero ahora «habla el idioma» del procesador.

## 6. Ejecución en un procesador cuántico real

¡El momento esperado! Enviamos el circuito de Bell a la QPU. El trabajo entra en una **cola** compartida con usuarios de todo el mundo, así que puede tardar desde segundos hasta varios minutos.

> 💡 Si la espera se alarga, guarda el ID del trabajo que se imprime. Más tarde puedes recuperar el resultado con `service.job("ID").result()`, incluso desde otra sesión.

In [ ]:
conteos_real = ejecutar_y_contar(bell, backend, shots=1000)
print(conteos_real)
plot_histogram(conteos_real, title=f"Estado de Bell en {backend.name}")

ID del trabajo: dbg2hrlrd2mc73ctr360


Para recuperar un trabajo

In [ ]:
#trabajo = service.job("   ")
#print(trabajo.status())    # QUEUED, RUNNING, DONE, ERROR...
#resultado = trabajo.result()

### 6.1 Simulador vs. hardware real: el efecto del ruido

En el hardware real aparecen resultados `01` y `10` que, en teoría, **nunca** deberían ocurrir. Son la huella del **ruido**:

- **Errores de compuerta:** las compuertas, sobre todo las de dos qubits, no son perfectas.
- **Errores de lectura:** a veces se mide 1 cuando el qubit estaba en 0, o viceversa.
- **Decoherencia:** el qubit pierde su información cuántica al interactuar con el entorno (tiempos $T_1$ y $T_2$).

Comparemos ambos histogramas lado a lado y cuantifiquemos el error.

In [ ]:
plot_histogram(
    [conteos_sim, conteos_real],
    legend=["Simulador ideal", f"QPU {backend.name}"],
    title="Estado de Bell: simulador vs. hardware real",
)

In [ ]:
def fraccion_correcta(conteos):
    # Fracción de shots con resultados permitidos por |Φ+⟩: '00' o '11'
    total = sum(conteos.values())
    return (conteos.get("00", 0) + conteos.get("11", 0)) / total

print(f"Simulador ideal: {fraccion_correcta(conteos_sim):.1%} de resultados correctos")
print(f"QPU real:        {fraccion_correcta(conteos_real):.1%} de resultados correctos")

### 6.2 (Opcional) Simular el ruido del hardware

Podemos construir un simulador que **imita el ruido** de la QPU que elegimos, a partir de sus datos de calibración. Es muy útil para prototipar sin gastar tiempo de QPU.

In [ ]:
# Simulador Aer con el modelo de ruido y la conectividad del backend real
simulador_ruidoso = AerSimulator.from_backend(backend)

conteos_ruido = ejecutar_y_contar(bell, simulador_ruidoso, shots=1000)

plot_histogram(
    [conteos_sim, conteos_ruido, conteos_real],
    legend=["Ideal", "Simulación con ruido", "QPU real"],
    title="Tres formas de ejecutar el mismo circuito",
)

## 7. Estimator: valor esperado de un Hamiltoniano

Muchas aplicaciones (química, materiales, optimización) no necesitan los conteos completos, sino el **valor esperado** de un **observable**:

$$
\langle O \rangle = \langle\psi|\,O\,|\psi\rangle .
$$

Para eso existe la primitiva **Estimator**.

### 7.1 El Hamiltoniano

Usaremos un modelo de **Ising con campo transversal** para dos espines:

$$
H = J\, Z_1 Z_0 + h_x\,(X_1 + X_0), \qquad J = 1,\quad h_x = -0.5 .
$$

- El término $J\,Z_1Z_0$ con $J>0$ es **antiferromagnético**: favorece que los espines apunten en direcciones opuestas.
- El término $h_x(X_1+X_0)$ es un **campo transversal** que intenta voltear los espines.

En Qiskit los observables se escriben como sumas de **cadenas de Pauli** con `SparsePauliOp`. Recuerda el orden little-endian: en `"XI"`, la `X` actúa sobre el qubit 1 y la `I` sobre el qubit 0.

In [ ]:
# Parámetros del modelo
J = 1.0     # acoplamiento antiferromagnético (J > 0)
hx = -0.5   # intensidad del campo transversal

# H = J Z1 Z0 + hx (X1 + X0)
H = SparsePauliOp.from_list([("ZZ", J), ("XI", hx), ("IX", hx)])
print(H)

### 7.2 El estado de prueba

Partimos del estado de Bell y aplicamos $X$ sobre $q_1$ y $Z$ sobre $q_0$:

$$
|\Phi^+\rangle \xrightarrow{X_1} \frac{|10\rangle + |01\rangle}{\sqrt2} \xrightarrow{Z_0} \frac{|10\rangle - |01\rangle}{\sqrt2} = -\,|\Psi^-\rangle .
$$

Es el **singlete** $|\Psi^-\rangle$ (salvo una fase global, que no tiene efecto físico): los dos espines son siempre opuestos. Fíjate en que este circuito **no** lleva mediciones: el Estimator se encarga de medir en las bases adecuadas.

In [ ]:
# Preparamos el estado (sin mediciones)
qc_singlete = QuantumCircuit(2)
qc_singlete.h(0)
qc_singlete.cx(0, 1)
qc_singlete.x(1)
qc_singlete.z(0)

display(Statevector(qc_singlete).draw("latex"))
qc_singlete.draw("mpl")

### 7.3 Primero, el valor exacto

Con dos qubits podemos calcular $\langle H\rangle$ exactamente desde el vector de estado. Nos servirá de **referencia** para evaluar al hardware.

Para el singlete, $\langle Z_1Z_0\rangle = -1$ (espines opuestos) y $\langle X_1\rangle = \langle X_0\rangle = 0$, así que esperamos $\langle H\rangle = -J = -1$.

In [ ]:
psi = Statevector(qc_singlete)

print("⟨Z1 Z0⟩ =", psi.expectation_value(SparsePauliOp("ZZ")).real)
print("⟨X1⟩    =", psi.expectation_value(SparsePauliOp("XI")).real)
print("⟨X0⟩    =", psi.expectation_value(SparsePauliOp("IX")).real)

E_exacta = psi.expectation_value(H).real
print("\n⟨H⟩ exacto =", E_exacta)

### 7.4 Estimator en el simulador y en la QPU

El patrón es el mismo que con el Sampler, con un detalle importante: al transpilar, los qubits virtuales se asignan a qubits físicos, así que el **observable también debe reubicarse** con `apply_layout`.

A la primitiva se le pasa un **PUB** (*Primitive Unified Bloc*): una tupla `(circuito, observable)`.

In [ ]:
def estimar(circuito, observable, backend, optimization_level=3):
    """Transpila el circuito, reubica el observable y devuelve (valor esperado, desviación estándar)."""
    pm = generate_preset_pass_manager(backend=backend, optimization_level=optimization_level)
    circuito_isa = pm.run(circuito)
    observable_isa = observable.apply_layout(layout=circuito_isa.layout)

    estimator = Estimator(mode=backend)
    trabajo = estimator.run([(circuito_isa, observable_isa)])
    print("ID del trabajo:", trabajo.job_id())
    resultado = trabajo.result()[0]
    return float(resultado.data.evs), float(resultado.data.stds)

In [ ]:
# Simulador ideal
E_sim, std_sim = estimar(qc_singlete, H, simulador)
print(f"Simulador ideal: ⟨H⟩ = {E_sim:.4f} ± {std_sim:.4f}")

In [ ]:
# Procesador cuántico real
E_real, std_real = estimar(qc_singlete, H, backend)
print(f"QPU {backend.name}: ⟨H⟩ = {E_real:.4f} ± {std_real:.4f}")

In [ ]:
# Resumen de la comparación
etiquetas = ["Exacto", "Simulador", f"QPU\n{backend.name}"]
valores = [E_exacta, E_sim, E_real]
errores = [0, std_sim, std_real]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(etiquetas, valores, yerr=errores, capsize=6, color=["gray", "tab:blue", "tab:purple"])
ax.axhline(E_exacta, ls="--", color="k", lw=1)
ax.set_ylabel(r"$\langle H \rangle$")
ax.set_title("Valor esperado del Hamiltoniano en el singlete")
plt.show()

print(f"Error relativo en la QPU: {abs(E_real - E_exacta) / abs(E_exacta):.1%}")

> **Sobre la mitigación de errores:** por defecto, el Estimator de IBM Runtime aplica técnicas de **mitigación de errores de lectura** (`resilience_level=1`). Por eso el valor obtenido en hardware suele estar más cerca del exacto de lo que sugeriría el histograma ruidoso del Sampler. Puedes explorar otros niveles con `estimator.options.resilience_level = 0` (sin mitigación) o `2` (más mitigación, más costosa).

### 7.5 ¿Es el singlete el estado de mínima energía?

El valor más bajo posible de $\langle H\rangle$ es el menor autovalor de $H$ (la **energía del estado fundamental**). Con dos qubits podemos diagonalizar la matriz $4\times4$ directamente:

In [ ]:
autovalores = np.linalg.eigvalsh(H.to_matrix())
print("Autovalores de H:", np.round(autovalores, 4))
print(f"Energía del estado fundamental: {autovalores[0]:.4f}")
print(f"Energía del singlete:           {E_exacta:.4f}")

¡El singlete **no** es el estado fundamental! Hay un estado con energía $-\sqrt2 \approx -1.414$. Encontrar estados de mínima energía en sistemas grandes, donde ya no podemos diagonalizar, es justamente el objetivo de algoritmos variacionales como el **VQE** (*Variational Quantum Eigensolver*). Veamos una versión en miniatura.

### 7.6 Mini-VQE: buscando el estado fundamental

Usamos un circuito con un **parámetro** $\theta$ y medimos la energía para muchos valores de $\theta$. Este circuito prepara

$$
|\psi(\theta)\rangle = (H\otimes H)\left[\cos\tfrac{\theta}{2}|00\rangle + \sin\tfrac{\theta}{2}|11\rangle\right],
$$

y se puede mostrar que su energía es $E(\theta) = \sin\theta - \cos\theta$, cuyo mínimo es $-\sqrt2$ en $\theta = -\pi/4$.

Gracias a los PUB, el Estimator evalúa **todos los valores del parámetro en un solo trabajo**.

In [ ]:
from qiskit.circuit import Parameter

theta = Parameter("θ")

ansatz = QuantumCircuit(2)
ansatz.ry(theta, 0)
ansatz.cx(0, 1)
ansatz.h([0, 1])
ansatz.draw("mpl")

In [ ]:
# Barrido de θ en el simulador: un único PUB con 41 valores del parámetro
valores_theta = np.linspace(-np.pi, np.pi, 41)

pm = generate_preset_pass_manager(backend=simulador, optimization_level=1)
ansatz_isa = pm.run(ansatz)
H_isa = H.apply_layout(ansatz_isa.layout)

estimator_sim = Estimator(mode=simulador)
res = estimator_sim.run([(ansatz_isa, H_isa, valores_theta.reshape(-1, 1))]).result()[0]
energias = res.data.evs

i_min = np.argmin(energias)
print(f"Energía mínima encontrada: {energias[i_min]:.4f} en θ = {valores_theta[i_min]:.3f} rad")
print(f"Valor exacto:              {autovalores[0]:.4f} en θ = {-np.pi/4:.3f} rad")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(valores_theta, energias, "o", label="Estimator (simulador)")
ax.plot(valores_theta, np.sin(valores_theta) - np.cos(valores_theta), "-", label=r"Teoría: $\sin\theta-\cos\theta$")
ax.axhline(autovalores[0], ls="--", color="k", lw=1, label="Estado fundamental")
ax.axhline(E_exacta, ls=":", color="tab:red", lw=1, label="Singlete")
ax.set_xlabel(r"$\theta$ (rad)")
ax.set_ylabel(r"$\langle H \rangle$")
ax.legend()
plt.show()

> **Para ir más lejos:** en lugar de un barrido, un VQE real usa un **optimizador clásico** (por ejemplo `scipy.optimize.minimize`) que propone nuevos valores de $\theta$ en un ciclo híbrido cuántico-clásico. ¡Puedes intentarlo en el Reto 4!

## 8. Retos para seguir explorando

**Reto 1: los cuatro estados de Bell.** Además de $|\Phi^+\rangle$ existen
$|\Phi^-\rangle = \tfrac{1}{\sqrt2}(|00\rangle - |11\rangle)$,
$|\Psi^+\rangle = \tfrac{1}{\sqrt2}(|01\rangle + |10\rangle)$ y
$|\Psi^-\rangle = \tfrac{1}{\sqrt2}(|01\rangle - |10\rangle)$.
Construye los cuatro añadiendo compuertas `x` y `z` al circuito de Bell. ¿Cuáles dan el mismo histograma? ¿Cómo los distinguirías? (Pista: mide en otra base aplicando `h` a ambos qubits antes de medir.)

**Reto 2: estado GHZ.** Generaliza el estado de Bell a tres qubits: $\tfrac{1}{\sqrt2}(|000\rangle + |111\rangle)$. Ejecútalo en el simulador y en la QPU. ¿La fracción de resultados correctos sube o baja respecto al de dos qubits? ¿Por qué?

**Reto 3: niveles de optimización.** Transpila un circuito GHZ de 5 qubits con `optimization_level` de 0 a 3 y compara la profundidad y el número de compuertas de dos qubits (`count_ops()`).

**Reto 4: VQE de verdad.** Usa `scipy.optimize.minimize` con el método `COBYLA` para encontrar el $\theta$ que minimiza la energía, llamando al Estimator dentro de la función objetivo.

**Reto 5: cambia la física.** Modifica $h_x$ y repite la diagonalización de la sección 7.5. ¿Para qué valores del campo el singlete sí es el estado fundamental?

In [ ]:
# Espacio para tus experimentos

## Contexto: una década de computación cuántica en la nube

En mayo de 2016 IBM puso por primera vez un procesador cuántico de **5 qubits** a disposición de cualquier persona a través de internet. Diez años después, en este taller has ejecutado un programa en un procesador de más de **100 qubits** desde un navegador.

### Lo que hicimos hoy

- Representamos qubits con **vectores de estado** y vimos **superposición**, **medición** e **interferencia**.
- Construimos el **estado de Bell** y entendimos el **entrelazamiento** a través de las correlaciones.
- Usamos la primitiva **Sampler** en un simulador y en una **QPU real**, y cuantificamos el efecto del **ruido**.
- Vimos cómo la **transpilación** adapta un circuito a las compuertas y conectividad del hardware.
- Calculamos el **valor esperado de un Hamiltoniano** con **Estimator** y dimos un primer paso hacia el **VQE**.

### Para seguir aprendiendo

- [IBM Quantum Learning](https://quantum.cloud.ibm.com/learning): cursos gratuitos, entre ellos *Basics of Quantum Information* y *Quantum Computing in Practice*.
- [Documentación de Qiskit](https://quantum.cloud.ibm.com/docs): guías y referencia de la API.
- Comunidad Qiskit: eventos, *Fall Fests* y programas de embajadores.

¡Gracias por participar! 🎉